In [17]:
import os
import re
import math
from collections import Counter
import pandas as pd

# Đường dẫn gốc tới corpus
root_path = r"D:\RUB\Research 2\vietnamese-lexical-decision-task\data_sample\Doi song"
out_path = r"D:\RUB\Research 2\vietnamese-lexical-decision-task\data_sample"

max_files_per_folder = 200

# Một số dòng/nhãn kiểu công thức, giá cả, nguyên liệu... không có ích cho scope project
# Có thể mở rộng thêm sau khi xem dữ liệu thật nhiều hơn
bad_line_starts = {
    "nguyên liệu",
    "thực hiện",
    "giá tham khảo",
}

# Những từ/dấu hiệu thường cho thấy đây là metadata hoặc web noise
bad_line_contains = {
    "vnexpress",
    "subject",
    "sent",
    "from:",
    "to:",
    "http",
    "https",
    "www.",
    ".com",
    ".vn",
    "@"
}

def clean_line_for_project(line):
    """
    Clean một dòng theo hướng có lợi cho project:
    - giữ tiếng Việt có dấu
    - bỏ số
    - bỏ dấu câu / ký hiệu
    - giữ khoảng trắng để bảo toàn syllable structure
    """
    line = line.strip().lower()

    # Bỏ các dòng rỗng
    if not line:
        return ""

    # Bỏ một số dòng tiêu đề kỹ thuật không có ích cho lexical decision stimuli
    if line in bad_line_starts or any(line.startswith(x + ":") for x in bad_line_starts):
        return ""

    # Bỏ số
    line = re.sub(r"\d+", " ", line)

    # Đổi dấu gạch nối thành khoảng trắng để tránh dính token
    line = line.replace("-", " ")

    # Giữ chữ cái, số gạch dưới mặc định của \w, khoảng trắng và chữ tiếng Việt có dấu
    # Mục tiêu ở đây là bỏ dấu câu/ký hiệu, không đụng vào chữ tiếng Việt
    line = re.sub(
        r"[^\w\sàáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ]",
        " ",
        line
    )

    # Bỏ dấu gạch dưới nếu có
    line = line.replace("_", " ")

    # Chuẩn hóa nhiều khoảng trắng liên tiếp
    line = re.sub(r"\s+", " ", line).strip()
    
    # bỏ dòng chứa HTML/web noise
    if any(x in line for x in ["img", "http", "www", "src", "width", "height", "border"]):
        return ""

    if any(x in line for x in bad_line_contains):
        return ""
    
    return line

def generate_ngrams(tokens, n):
    """
    Tạo n-gram liên tiếp từ danh sách syllables.
    Ví dụ:
    ['học', 'sinh', 'giỏi'] với n=2
    -> ['học sinh', 'sinh giỏi']
    """
    return [" ".join(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

# stopwords để loại function words
stopwords = {
    "và", "của", "là", "có", "cho", "các", "trong", "với",
    "được", "những", "một", "đã", "sẽ", "đến", "không",
    "như", "cũng", "này", "đó", "theo", "rằng", "tôi",
    "anh", "ta", "ấy", "chúng"
}

bad_tokens = {
    "vnexpress", "net", "sent", "subject", "to", "from",
    "am", "pm", "com", "www", "http", "https"
}

# kiểm tra ngram có hợp lệ không
def is_valid_ngram(ngram):
    words = ngram.split()
    return not any(w in stopwords for w in words)

all_unigrams = []
all_bigrams = []
all_trigrams = []

file_count = 0

for root, dirs, files in os.walk(root_path):

    file_counter = 0  # reset cho mỗi folder

    for file in files:
        if file.endswith(".txt"):

            if file_counter >= max_files_per_folder:
                break

            file_path = os.path.join(root, file)
            file_counter += 1
            file_count += 1

            # đọc file với fallback encoding để tránh crash
            try:
                with open(file_path, "r", encoding="utf-8") as f:
                    lines = f.readlines()
            except UnicodeDecodeError:
                try:
                    with open(file_path, "r", encoding="utf-16") as f:
                        lines = f.readlines()
                except:
                    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                        lines = f.readlines()

            cleaned_lines = []

            for line in lines:
                clean_line = clean_line_for_project(line)
                if clean_line:
                    cleaned_lines.append(clean_line)

            text = " ".join(cleaned_lines)
            tokens = text.split()

            # Bỏ token metadata / web noise còn sót lại
            tokens = [t for t in tokens if t not in bad_tokens]

            if not tokens:
                continue

            all_unigrams.extend(tokens)
            bigrams = generate_ngrams(tokens, 2)
            trigrams = generate_ngrams(tokens, 3)
            
            # lọc stopwords
            bigrams = [b for b in bigrams if is_valid_ngram(b)]
            trigrams = [t for t in trigrams if is_valid_ngram(t)]
            
            all_bigrams.extend(bigrams)
            all_trigrams.extend(trigrams)

print(f"So file da doc: {file_count}")
print(f"So syllable tokens: {len(all_unigrams)}")
print(f"So bigrams: {len(all_bigrams)}")
print(f"So trigrams: {len(all_trigrams)}")

# Tính frequency
freq_uni = Counter(all_unigrams)
freq_bi = Counter(all_bigrams)
freq_tri = Counter(all_trigrams)

# Đổi sang DataFrame để dễ nhìn và lọc
df_uni = pd.DataFrame(freq_uni.items(), columns=["lexical_item", "frequency"])
df_bi = pd.DataFrame(freq_bi.items(), columns=["lexical_item", "frequency"])
df_tri = pd.DataFrame(freq_tri.items(), columns=["lexical_item", "frequency"])

# Gắn số syllables
df_uni["syllable_length"] = 1
df_bi["syllable_length"] = 2
df_tri["syllable_length"] = 3

# Log frequency để chuẩn bị cho analysis sau này
df_uni["log_frequency"] = df_uni["frequency"].apply(lambda x: math.log(x))
df_bi["log_frequency"] = df_bi["frequency"].apply(lambda x: math.log(x))
df_tri["log_frequency"] = df_tri["frequency"].apply(lambda x: math.log(x))

# Gộp lại thành một bảng chung
df_all = pd.concat([df_uni, df_bi, df_tri], ignore_index=True)

# Sắp xếp giảm dần theo frequency
df_all = df_all.sort_values(by=["syllable_length", "frequency"], ascending=[True, False])

# Lưu output
output_dir = os.path.join(out_path, "_processed")
os.makedirs(output_dir, exist_ok=True)

df_uni.to_csv(os.path.join(output_dir, "unigrams_frequency.csv"), index=False, encoding="utf-8-sig")
df_bi.to_csv(os.path.join(output_dir, "bigrams_frequency.csv"), index=False, encoding="utf-8-sig")
df_tri.to_csv(os.path.join(output_dir, "trigrams_frequency.csv"), index=False, encoding="utf-8-sig")
df_all.to_csv(os.path.join(output_dir, "all_1to3gram_frequency.csv"), index=False, encoding="utf-8-sig")

print("\nTop 20 unigrams:")
print(df_uni.sort_values("frequency", ascending=False).head(20).to_string(index=False))

print("\nTop 20 bigrams:")
print(df_bi.sort_values("frequency", ascending=False).head(20).to_string(index=False))

print("\nTop 20 trigrams:")
print(df_tri.sort_values("frequency", ascending=False).head(20).to_string(index=False))

print(f"\nDa luu file vao: {output_dir}")

So file da doc: 200
So syllable tokens: 91013
So bigrams: 56664
So trigrams: 44267

Top 20 unigrams:
lexical_item  frequency  syllable_length  log_frequency
          có       1639                1       7.401842
         anh       1604                1       7.380256
       không       1506                1       7.317212
         tôi       1410                1       7.251345
          và       1278                1       7.153052
          là       1155                1       7.051856
       người       1112                1       7.013915
         của       1110                1       7.012115
         với        860                1       6.756932
         một        851                1       6.746412
         bạn        830                1       6.721426
         cho        800                1       6.684612
         yêu        744                1       6.612041
        được        727                1       6.588926
       những        660                1       6.492240
   

In [24]:
import requests
from bs4 import BeautifulSoup
import re
import os


def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\sÀ-ỹ]', '', text)
    return text.lower().strip()


def extract_text_from_url(url):
    try:
        res = requests.get(url, timeout=10)
        soup = BeautifulSoup(res.text, "html.parser")

        article = soup.find("article")
        if article:
            text = article.get_text()
        else:
            text = soup.get_text()

        return clean_text(text)

    except Exception as e:
        print(f"Error with {url}: {e}")
        return ""


def save_to_txt(text, filename, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    path = os.path.join(output_dir, filename)

    with open(path, "w", encoding="utf-8") as f:
        f.write(text)

    return path

urls = [
    "https://vnexpress.net/oto-tai-tong-5-xe-may-o-ca-mau-5057797.html"
]

output_dir = r"D:\RUB\Research 2\vietnamese-lexical-decision-task"


for i, url in enumerate(urls):
    print(f"Processing {i+1}/{len(urls)}: {url}")

    text = extract_text_from_url(url)

    if text:
        filename = f"article_{i+1}.txt"
        path = save_to_txt(text, filename, output_dir)
        print(f"Saved: {path}")
    else:
        print("Skipped (empty)")

Processing 1/1: https://vnexpress.net/oto-tai-tong-5-xe-may-o-ca-mau-5057797.html
Saved: D:\RUB\Research 2\vietnamese-lexical-decision-task\article_1.txt
